## ????

????:http://algo.tpai.qq.com/home/information/index.html

????????????
![](http://static.zybuluo.com/zhuanxu/evhuwbyqmrvfsvuypsbfrry6/image_1cak7ik2ituh32giqnb65br09.png)

In [ ]:
import zipfile
import numpy as np
import pandas as pd

In [ ]:
data_root = "./pre"
# load data
dfTrain = pd.read_csv("%s/train.csv"%data_root)
dfTest = pd.read_csv("%s/test.csv"%data_root)
dfAd = pd.read_csv("%s/ad.csv"%data_root)

??????(train.csv),??????????,??????????,?????:?label,clickTime,conversionTime,creativeID,userID,positionID,connectionType,telecomsOperator??

?label=0?,conversionTime????????

?:??????0???????????(????ID(sitesetID)?0??????,????????????)


In [ ]:
dfTrain.head()

In [ ]:
dfTrain[dfTrain['label']==1].head()

??????? clickTime????

In [ ]:
dfTrain['clickTime'].min(),dfTrain['clickTime'].max()

clickTime,conversionTime,installTime,????DDHHMM,??DD?????,HH????,MM?????

??????????????,?????????????,??????30??????val??

In [ ]:
dfTrain['day'] = dfTrain['clickTime'].apply(lambda t : int(t/10000))

In [ ]:
a = dfTrain.groupby('day').label.agg(['mean','size'])

In [ ]:
a

In [ ]:
train = dfTrain[dfTrain['day']<30].reset_index()
val   = dfTrain[dfTrain['day'] == 30].reset_index()

In [ ]:
dfAd.head()

## Ad????
????????,???lr,????????,????????appId????,?????????

In [ ]:
# process data
trainAll = pd.merge(train, dfAd, on="creativeID")
valAll = pd.merge(val, dfAd, on="creativeID")
y_train = trainAll["label"].values

In [ ]:
key = "appID"
dfCvr = trainAll.groupby(key).apply(lambda df: np.mean(df["label"])).reset_index()

In [ ]:
dfCvr.head()

In [ ]:
dfCvr.columns = ['appID','prior_cvr'] # ????

In [ ]:
dfVal = pd.merge(valAll, dfCvr, how="left", on=key)

In [ ]:
dfVal[['label','prior_cvr']].head()

In [ ]:
# ?????appId????? train?,????na?,???????????????????
dfVal[dfVal['prior_cvr'].isna()]['appID'].unique()

In [ ]:
dfVal["prior_cvr"].fillna(np.mean(trainAll["label"]), inplace=True)

?? dfVal ?logloss?

In [ ]:
from sklearn.metrics import log_loss

In [ ]:
# ?????????? 0.09
log_loss(dfVal['label'].values, dfVal['prior_cvr'].values)

## lr ????

In [ ]:
trainAll.head()

??????????? Lr,???? ????????????encode

In [ ]:
from scipy import sparse
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

In [ ]:
# feature engineering/encoding
enc = OneHotEncoder()
# ??????user_id??,????????
feats = ["creativeID", "adID", "camgaignID", "advertiserID", "appID", "appPlatform"]
for i,feat in enumerate(feats):
    x_train = enc.fit_transform(trainAll[feat].values.reshape(-1, 1))
    # ??,???? val ?????? train ????,??????0
    X_val = enc.transform(valAll[feat].values.reshape(-1, 1))
    if i == 0:
        X_train, X_val = x_train, x_val
    else:
        X_train, X_val = sparse.hstack((X_train, x_train)), sparse.hstack((X_val, x_val))

In [ ]:
# model training
lr = LogisticRegression()
lr.fit(X_train, y_train)
proba_val = lr.predict_proba(X_val)[:,1]

In [ ]:
# ?????????? 0.087
log_loss(dfVal['label'].values, proba_val)

In [ ]:
import sklearn.externals.joblib as jl

In [ ]:
jl.dump(trainAll,"%s/trainAll.pkl"%data_root)
jl.dump(valAll,"%s/valAll.pkl"%data_root)